# 02 — Content-Based Recommendation with TF-IDF

**Purpose:** Build an item-feature representation from MovieLens movie titles and genre tags, then rank similar movies with cosine similarity.

This notebook is a reproducible research implementation. It uses only metadata present in MovieLens 1M; it does not pretend MovieLens contains plot summaries or cast data.

## 1. Load movies and prepare text features

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
# Works when launched from the Spectra root or from the notebooks folder.
if not (PROJECT_ROOT / "datasets" / "ml-1m").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "datasets" / "ml-1m"

movies = pd.read_csv(
    DATA_DIR / "movies.dat", sep="::", engine="python",
    names=["movie_id", "title", "genres"], encoding="latin-1"
)
ratings = pd.read_csv(
    DATA_DIR / "ratings.dat", sep="::", engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"], encoding="latin-1"
)
users = pd.read_csv(
    DATA_DIR / "users.dat", sep="::", engine="python",
    names=["user_id", "gender", "age", "occupation", "zip_code"], encoding="latin-1"
)
print(f"Movies: {len(movies):,} | Ratings: {len(ratings):,} | Users: {len(users):,}")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = movies.copy()
movies["feature_text"] = (
    movies["title"].fillna("").str.replace(r"\(\d{4}\)", "", regex=True)
    + " " + movies["genres"].fillna("").str.replace("|", " ", regex=False)
)
movies["feature_text"] = movies["feature_text"].str.lower().str.strip()
movies[["movie_id", "title", "genres", "feature_text"]].head()

Movies: 3,883 | Ratings: 1,000,209 | Users: 6,040


,movie_id,title,genres,feature_text
0,1,Toy Story (1995),Animation|Children's|Comedy,toy story animation children's comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy,jumanji adventure children's fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,grumpier old men comedy romance
3,4,Waiting to Exhale (1995),Comedy|Drama,waiting to exhale comedy drama
4,5,Father of the Bride Part II (1995),Comedy,father of the bride part ii comedy


## 2. TF-IDF vectorisation

In [2]:
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_features=20000
)
item_matrix = tfidf.fit_transform(movies["feature_text"])
print("TF-IDF matrix shape:", item_matrix.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

TF-IDF matrix shape: (3883, 11703)
Vocabulary size: 11703


## 3. Similarity search

In [3]:
movie_id_to_idx = pd.Series(movies.index, index=movies.movie_id).to_dict()
idx_to_movie = movies.set_index("movie_id")

def recommend_similar(movie_id, top_n=10):
    if movie_id not in movie_id_to_idx:
        raise KeyError(f"Unknown MovieLens movie_id: {movie_id}")
    idx = movie_id_to_idx[movie_id]
    scores = cosine_similarity(item_matrix[idx], item_matrix).ravel()
    scores[idx] = -1
    best = np.argsort(scores)[::-1][:top_n]
    result = movies.iloc[best][["movie_id", "title", "genres"]].copy()
    result["similarity"] = scores[best]
    return result.reset_index(drop=True)

# Example: MovieLens ID 1 is Toy Story (1995).
recommend_similar(1, top_n=10)

,movie_id,title,genres,similarity
0,3114,Toy Story 2 (1999),Animation|Children's|Comedy,1.000000
1,3400,We're Back! A Dinosaur's Story (1993),Animation|Children's,0.506608
2,588,Aladdin (1992),Animation|Children's|Comedy|Musical,0.288215
3,2354,"Rugrats Movie, The (1998)",Animation|Children's|Comedy,0.266632
4,2141,"American Tail, An (1986)",Animation|Children's|Comedy,0.264280
5,2355,"Bug's Life, A (1998)",Animation|Children's|Comedy,0.263554
6,1566,Hercules (1997),Adventure|Animation|Children's|Comedy|Musical,0.260235
7,2559,"King and I, The (1999)",Animation|Children's,0.257770
8,239,"Goofy Movie, A (1995)",Animation|Children's|Comedy|Romance,0.255988
9,3751,Chicken Run (2000),Animation|Children's|Comedy,0.253849


## 4. Inspect feature weights

In [4]:
feature_names = np.array(tfidf.get_feature_names_out())
toy_idx = movie_id_to_idx[1]
row = item_matrix[toy_idx].toarray().ravel()
top_features = np.argsort(row)[::-1][:15]
pd.DataFrame({"feature": feature_names[top_features], "tfidf_weight": row[top_features]})

,feature,tfidf_weight
0,toy,0.460465
1,toy story,0.460465
2,story animation,0.444243
3,story,0.328778
4,animation children,0.271901
5,children comedy,0.266226
6,animation,0.259451
7,children,0.208211
8,comedy,0.122571
9,émile,0.000000


## 5. Limitations and interpretation

- Genre tags and title tokens provide a relatively shallow representation.
- TF-IDF does not understand plot semantics, themes or cast relationships.
- Similarity is based on metadata overlap, not user preference.
- TMDB metadata can later enrich this feature space, but that integration must be evaluated separately.

Record examples where recommendations are sensible and where they fail.